In [2]:
import Pkg
Pkg.activate("/Users/mckinleypaul/Documents/montecarlo/gc_wl")
using JLD2
using Statistics
using LinearAlgebra
using Plots
using Plots.Measures  # for mm units in plot margins
using Printf
using LaTeXStrings


  Activating project at `~/Documents/montecarlo/gc_wl`


In [3]:

# ═══════════════════════════════════════════════════════════════════════════
#  Visualization suite for the 100k benchmark campaigns at L=5 and L=20.
#  Each plot is annotated with the statistical concept it demonstrates.
#
#  Structure:
#    Part 1: Fundamentals (distributions, CLT, SEM scaling)
#    Part 2: Estimator bias (Jensen gap)
#    Part 3: Noise propagation through the pipeline
#    Part 4: Correlation structure
#    Part 5: L-dependence (the physical signal question)
# ═══════════════════════════════════════════════════════════════════════════

# ─── paths — EDIT THESE to point to your two .jld2 campaign outputs ───
const L20_PATH = "/Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/100k_run/correlation_campaign_100000runs.jld2"
const L05_PATH = "/Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/100k_run_L5/correlation_campaign_100000runs.jld2"
const OUTDIR = "/Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/plots"
mkpath(OUTDIR)

# Load both campaigns
d20 = JLD2.load(L20_PATH)
d05 = JLD2.load(L05_PATH)

# Convenience accessors
N_max = 13
n_runs = d20["n_runs"]

# Global plot defaults for consistent aesthetics
default(fontfamily="Helvetica", framestyle=:box, grid=true, gridalpha=0.3,
    tickfontsize=9, labelfontsize=10, titlefontsize=11, legendfontsize=9,
    margin=3mm, dpi=150)

# Helper to save each figure with an informative filename
savefig_at(p, name) = savefig(p, joinpath(OUTDIR, name))


savefig_at (generic function with 1 method)

In [4]:

# ═══════════════════════════════════════════════════════════════════════════
#  PART 1: FUNDAMENTALS — distributions, central limit theorem, SEM scaling
# ═══════════════════════════════════════════════════════════════════════════

# ────────── Plot 1.1: Distribution of per-run B_n values ──────────
# CONCEPT: The central limit theorem guarantees that the mean of independent
# samples from any distribution with finite variance approaches Gaussian. Here
# each B_n per-run value is itself the result of a complicated Monte Carlo
# process, but the empirical distribution across 100k runs should look close
# to Gaussian if the per-run estimator has finite variance and the 100k runs
# are effectively independent. The width of each histogram is the per-run std.
plot_11_a = plot(layout=(3, 4), size=(1400, 900),
    plot_title="Per-run B_n distributions across 100,000 runs (L=20)\nEach panel: histogram of that Bn across all runs. Gaussian-like → central limit theorem applies",
    plot_titlefontsize=11)
for (idx, n) in enumerate(2:13)
    data = d20["Bns_store"][n, :]
    μ, σ = mean(data), std(data)
    histogram!(plot_11_a, data, subplot=idx,
        bins=60, normalize=:pdf, linecolor=:black, linewidth=0.2,
        fillcolor=:steelblue, fillalpha=0.6, label="")
    # Overlay Gaussian fit
    xs = range(μ - 4σ, μ + 4σ, length=200)
    ys = @. exp(-(xs - μ)^2 / (2σ^2)) / (σ * sqrt(2π))
    plot!(plot_11_a, xs, ys, subplot=idx, linecolor=:crimson, linewidth=2,
        label="Gaussian fit")
    title!(plot_11_a, "B_$n  (μ=$(round(μ, sigdigits=4)), σ=$(round(σ, sigdigits=3)))",
        subplot=idx)
end
savefig_at(plot_11_a, "01_1_per_run_Bn_distributions_L20.png")

# Same for L=5
plot_11_b = plot(layout=(3, 4), size=(1400, 900),
    plot_title="Per-run B_n distributions across 100,000 runs (L=5)",
    plot_titlefontsize=11)
for (idx, n) in enumerate(2:13)
    data = d05["Bns_store"][n, :]
    μ, σ = mean(data), std(data)
    histogram!(plot_11_b, data, subplot=idx,
        bins=60, normalize=:pdf, linecolor=:black, linewidth=0.2,
        fillcolor=:darkorange, fillalpha=0.6, label="")
    xs = range(μ - 4σ, μ + 4σ, length=200)
    ys = @. exp(-(xs - μ)^2 / (2σ^2)) / (σ * sqrt(2π))
    plot!(plot_11_b, xs, ys, subplot=idx, linecolor=:crimson, linewidth=2,
        label="Gaussian fit")
    title!(plot_11_b, "B_$n  (μ=$(round(μ, sigdigits=4)), σ=$(round(σ, sigdigits=3)))",
        subplot=idx)
end
savefig_at(plot_11_b, "01_2_per_run_Bn_distributions_L05.png")

# ────────── Plot 1.2: SEM scaling with k (1/√k check) ──────────
# CONCEPT: The standard error of the mean (SEM) of k i.i.d. samples scales as
# σ/√k. On a log-log plot, SEM vs k should fall on a line with slope −1/2.
# Deviations diagnose correlated samples (slope > −1/2) or other pathologies.
# This is the single most important sanity check on the whole workflow.
function plot_sem_scaling(d, Lstr)
    snap_ks = sort(collect(keys(d["snapshot_sem_Bns"])))
    p = plot(size=(900, 600), xscale=:log10, yscale=:log10,
        xlabel="number of runs k", ylabel="SEM(B_n)",
        title="SEM scaling vs k — should follow k^(−1/2)  ($Lstr)",
        legend=:outerright, legendtitle="coeff")
    for n in 2:13
        sems = [d["snapshot_sem_Bns"][k][n] for k in snap_ks]
        plot!(p, snap_ks, sems, marker=:circle, markersize=4,
            label="B_$n")
    end
    # Reference 1/√k line anchored at first snapshot for B_13
    k_ref = snap_ks
    ref_anchor = d["snapshot_sem_Bns"][snap_ks[1]][13]
    plot!(p, k_ref, ref_anchor .* sqrt.(snap_ks[1] ./ k_ref),
        linestyle=:dash, linecolor=:black, linewidth=2,
        label="k^(−1/2) reference")
    return p
end
savefig_at(plot_sem_scaling(d20, "L=20"), "02_1_SEM_scaling_L20.png")
savefig_at(plot_sem_scaling(d05, "L=5"), "02_2_SEM_scaling_L05.png")

# ────────── Plot 1.3: Empirical vs theoretical SEM ──────────
# CONCEPT: The SEM estimate (std/√n_runs) at each k should match what you get
# from bootstrap / from CLT prediction. Plotting the ratio (empirical SEM at k)
# / (σ_full / √k) should be ≈ 1 for all k. Deviations from 1 indicate either
# variance itself varies with k (inconsistency) or correlations.
function plot_sem_ratio(d, Lstr)
    snap_ks = sort(collect(keys(d["snapshot_sem_Bns"])))
    p = plot(size=(900, 500), xscale=:log10,
        xlabel="number of runs k", ylabel="empirical SEM / (σ_100k / √k)",
        title="Ratio of SEM(k) to CLT prediction ($Lstr)\nShould be ≈ 1 for all k if samples are i.i.d.",
        legend=:outerright, ylims=(0.7, 1.3))
    for n in 2:13
        σ_full = d["Bns_std"][n]
        ratios = [d["snapshot_sem_Bns"][k][n] / (σ_full / sqrt(k)) for k in snap_ks]
        plot!(p, snap_ks, ratios, marker=:circle, markersize=4, label="B_$n")
    end
    hline!(p, [1.0], linecolor=:black, linewidth=2, linestyle=:dash, label="ideal")
    return p
end
savefig_at(plot_sem_ratio(d20, "L=20"), "03_1_SEM_ratio_L20.png")
savefig_at(plot_sem_ratio(d05, "L=5"), "03_2_SEM_ratio_L05.png")


"/Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/plots/03_2_SEM_ratio_L05.png"

In [5]:

# ═══════════════════════════════════════════════════════════════════════════
#  PART 2: ESTIMATOR BIAS — the Jensen gap story
# ═══════════════════════════════════════════════════════════════════════════

# ────────── Plot 2.1: Two estimators for B_n, side by side, with errors ──────────
# CONCEPT: Two ways to extract B_n: (a) compute B_n per run and average,
# (b) average logQ first and compute B_n from the mean. For nonlinear
# transformations (here exp() in the bn recursion), these give different
# answers — Jensen's inequality. Plotted here with error bars = SEM.
function plot_two_estimators_Bn(d, Lstr)
    ns = 2:13
    Bmean = d["Bns_mean"][ns]
    Bsem = d["Bns_sem"][ns]
    Bmlog = d["Bns_from_meanlogQ"][ns]

    p = plot(size=(900, 600), yscale=:log10,
        xlabel="n", ylabel="B_n",
        title="Two estimators of B_n at $Lstr — Jensen gap visible at high n",
        legend=:bottomright)
    plot!(p, ns, Bmean, yerror=Bsem, marker=:circle, markersize=6,
        linewidth=2, label="mean of per-run B_n", color=:steelblue)
    plot!(p, ns, Bmlog, marker=:square, markersize=6,
        linewidth=2, label="B_n from mean logQ", color=:crimson,
        linestyle=:dash)
    return p
end
savefig_at(plot_two_estimators_Bn(d20, "L=20"), "04_1_two_estimators_Bn_L20.png")
savefig_at(plot_two_estimators_Bn(d05, "L=5"), "04_2_two_estimators_Bn_L05.png")

# ────────── Plot 2.2: Jensen gap magnitude vs SEM, on one axis ──────────
# CONCEPT: The question is whether the Jensen gap is a small-and-ignorable
# systematic or a dominant-above-the-noise-floor bias. Plotting |gap| and SEM
# on the same axis shows at which n the gap exceeds SEM — i.e. where mean-of-
# per-run becomes demonstrably biased.
function plot_gap_vs_SEM_Bn(d, Lstr)
    ns = 2:13
    gap = abs.(d["Bns_jensen_gap"][ns])
    sem = d["Bns_sem"][ns]

    p = plot(size=(900, 600), yscale=:log10,
        xlabel="n", ylabel="magnitude",
        title="|Jensen gap| vs SEM(B_n) at $Lstr\nBias dominates noise where gap > SEM",
        legend=:topleft)
    plot!(p, ns, gap, marker=:circle, markersize=7, linewidth=2,
        label="|Jensen gap|", color=:crimson)
    plot!(p, ns, sem, marker=:square, markersize=7, linewidth=2,
        label="SEM", color=:steelblue)
    plot!(p, ns, 3 .* sem, linestyle=:dot, linewidth=2,
        label="3 × SEM (significance threshold)", color=:steelblue)
    return p
end
savefig_at(plot_gap_vs_SEM_Bn(d20, "L=20"), "05_1_gap_vs_SEM_L20.png")
savefig_at(plot_gap_vs_SEM_Bn(d05, "L=5"), "05_2_gap_vs_SEM_L05.png")

# ────────── Plot 2.3: gap/SEM growing with n (the bias accumulates) ──────────
# CONCEPT: gap/SEM is the "how many sigmas away" metric. Values > 3 indicate
# statistically significant bias. The plot shows how this accumulates with
# order: low-n coefficients are nearly unbiased, high-n coefficients are
# tens of sigmas off.
function plot_gap_over_SEM(d20, d05)
    ns = 2:13
    g20 = abs.(d20["Bns_jensen_over_sem"][ns])
    g05 = abs.(d05["Bns_jensen_over_sem"][ns])

    p = plot(size=(900, 500),
        xlabel="n", ylabel="|Jensen gap| / SEM (σ-units)",
        title="Jensen bias in units of SEM — bias dominates above ~3σ",
        legend=:topleft)
    plot!(p, ns, g20, marker=:circle, markersize=7, linewidth=2,
        label="L=20", color=:steelblue)
    plot!(p, ns, g05, marker=:square, markersize=7, linewidth=2,
        label="L=5", color=:darkorange)
    hline!(p, [3.0], linestyle=:dash, linecolor=:black,
        linewidth=2, label="3σ threshold")
    return p
end
savefig_at(plot_gap_over_SEM(d20, d05), "06_gap_over_SEM_both_L.png")

# ────────── Plot 2.4: Jensen gap as % of B_n itself ──────────
# CONCEPT: The fractional bias — gap as a percentage of B_n — tells us whether
# the bias matters in physical units. Compare to the fractional SEM. At L=5
# and L=20 the fractional Jensen gap on B_n is roughly L-independent; this
# suggests the bias is controlled by per-run logQ noise level (maxiter-
# determined), not by the physical regime.
function plot_gap_fraction(d20, d05)
    ns = 2:13
    f20 = abs.(d20["Bns_jensen_gap"][ns] ./ d20["Bns_from_meanlogQ"][ns])
    f05 = abs.(d05["Bns_jensen_gap"][ns] ./ d05["Bns_from_meanlogQ"][ns])

    p = plot(size=(900, 500), yscale=:log10,
        xlabel="n", ylabel="|Jensen gap| / |B_n|",
        title="Fractional Jensen bias on B_n — nearly L-independent\n" *
              "Fraction controlled by per-run logQ noise, not by V",
        legend=:bottomright)
    plot!(p, ns, f20, marker=:circle, markersize=7, linewidth=2,
        label="L=20", color=:steelblue)
    plot!(p, ns, f05, marker=:square, markersize=7, linewidth=2,
        label="L=5", color=:darkorange)
    return p
end
savefig_at(plot_gap_fraction(d20, d05), "07_fractional_bias_both_L.png")


┌ Warning: No strict ticks found
└ @ PlotUtils /Users/mckinleypaul/.julia/packages/PlotUtils/HX80C/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /Users/mckinleypaul/.julia/packages/PlotUtils/HX80C/src/ticks.jl:194
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/mckinleypaul/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: No strict ticks found
└ @ PlotUtils /Users/mckinleypaul/.julia/packages/PlotUtils/HX80C/src/ticks.jl:194


All plots saved to /Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/plots


┌ Warning: No strict ticks found
└ @ PlotUtils /Users/mckinleypaul/.julia/packages/PlotUtils/HX80C/src/ticks.jl:194
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/mckinleypaul/.julia/packages/Plots/GIume/src/utils.jl:105


Ordering: 00 is the summary; 01–19 walk from fundamentals through to advanced analysis.


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
#  PART 3: NOISE PROPAGATION THROUGH THE PIPELINE
# ═══════════════════════════════════════════════════════════════════════════

# ────────── Plot 3.1: Per-run std at each pipeline stage ──────────
# CONCEPT: How noise amplifies through logQ → bn → Bn. Per-run std of logQ is
# modest (~0.05-0.07). Per-run std of bn is enormous for high n (can exceed
# bn's mean). Per-run std of Bn is intermediate — the Hellmann-Bich polynomial
# partially cancels the correlated bn noise. This is the "counter-cancellation"
# story.
function plot_std_pipeline(d, Lstr)
    Ns = 0:13
    ns_b = 1:13
    ns_B = 2:13

    p1 = plot(Ns, d["logQ_std"], marker=:circle, linewidth=2,
        xlabel="N", ylabel="per-run std(logQ_N)",
        title="Stage 1: WL noise on logQ", label="", color=:darkgreen)

    p2 = plot(ns_b, d["bns_std"], marker=:circle, linewidth=2, yscale=:log10,
        xlabel="n", ylabel="per-run std(b_n)",
        title="Stage 2: Amplified through b_n recursion", label="",
        color=:steelblue)
    # Overlay |b_n| so we can see when std >> |mean|
    plot!(p2, ns_b, abs.(d["bns_mean"]), marker=:square, linewidth=2,
        linestyle=:dash, color=:crimson, label="|mean b_n|")
    p2 = plot!(p2, legend=:bottomleft)

    p3 = plot(ns_B, d["Bns_std"][ns_B], marker=:circle, linewidth=2, yscale=:log10,
        xlabel="n", ylabel="per-run std(B_n)",
        title="Stage 3: Partly uncancelled in B_n", label="",
        color=:steelblue)
    plot!(p3, ns_B, d["Bns_mean"][ns_B], marker=:square, linewidth=2,
        linestyle=:dash, color=:crimson, label="B_n mean")
    p3 = plot!(p3, legend=:topleft)

    p = plot(p1, p2, p3, layout=(1, 3), size=(1500, 450),
        plot_title="Noise propagation through the pipeline ($Lstr)",
        plot_titlefontsize=12, left_margin=6mm, bottom_margin=6mm)
    return p
end
savefig_at(plot_std_pipeline(d20, "L=20"), "08_1_noise_pipeline_L20.png")
savefig_at(plot_std_pipeline(d05, "L=5"), "08_2_noise_pipeline_L05.png")

# ────────── Plot 3.2: Signal-to-noise ratio per run, for b_n and B_n ──────────
# CONCEPT: SNR = |mean| / std per run. SNR > 1 means a single run gives you
# useful information about the true value; SNR < 1 means a single run is
# essentially noise. The striking result: for bn, single runs are garbage for
# n ≥ 3. For Bn, single runs give useful-but-noisy values through the whole
# range.
function plot_snr(d, Lstr)
    ns_b = 2:13
    ns_B = 2:13
    snr_b = abs.(d["bns_mean"][ns_b]) ./ d["bns_std"][ns_b]
    snr_B = abs.(d["Bns_mean"][ns_B]) ./ d["Bns_std"][ns_B]

    p = plot(size=(900, 600), yscale=:log10,
        xlabel="n", ylabel="per-run signal/noise = |mean| / std",
        title="Per-run signal-to-noise ratio ($Lstr)\n" *
              "SNR < 1 → single runs dominated by noise",
        legend=:topright)
    plot!(p, ns_b, snr_b, marker=:circle, markersize=7, linewidth=2,
        label="b_n", color=:steelblue)
    plot!(p, ns_B, snr_B, marker=:square, markersize=7, linewidth=2,
        label="B_n", color=:crimson)
    hline!(p, [1.0], linestyle=:dash, linecolor=:black, linewidth=2,
        label="SNR = 1")
    return p
end
savefig_at(plot_snr(d20, "L=20"), "09_1_SNR_per_run_L20.png")
savefig_at(plot_snr(d05, "L=5"), "09_2_SNR_per_run_L05.png")


In [6]:

# ═══════════════════════════════════════════════════════════════════════════
#  PART 4: CORRELATION STRUCTURE
# ═══════════════════════════════════════════════════════════════════════════

# ────────── Plot 4.1: logQ correlation matrix as a heatmap ──────────
# CONCEPT: WL samples logQ(N) by following a trajectory through N-space,
# so adjacent-N bins are highly correlated (0.95+ for nearest neighbors) and
# the correlation decays slowly with |ΔN|. This is trajectory-level coupling
# and is a property of the algorithm, not the physics.
function plot_logQ_corr(d, Lstr)
    M = d["logQ_corr"][2:end, 2:end]  # drop the N=0 NaN row/col
    labels = ["N=$N" for N in 1:13]
    p = heatmap(labels, labels, M, size=(700, 650),
        title="logQ correlation matrix ($Lstr)\n" *
              "WL trajectory induces strong nearest-neighbor coupling",
        xlabel="", ylabel="", c=:RdBu, clims=(-1, 1),
        yflip=true, aspect_ratio=:equal)
    return p
end
savefig_at(plot_logQ_corr(d20, "L=20"), "10_1_logQ_corr_L20.png")
savefig_at(plot_logQ_corr(d05, "L=5"), "10_2_logQ_corr_L05.png")

# ────────── Plot 4.2: bn correlation matrix ──────────
# CONCEPT: bn's are tridiagonally anticorrelated: each bn is anticorrelated
# ≈ −0.5 with its nearest neighbors, essentially zero with more distant ones.
# This is the "recursion subtracts bn−1 to produce bn" structure propagating
# through the Ushcats recursion. It means bn noise has tridiagonal covariance.
function plot_bns_corr(d, Lstr)
    M = d["bns_corr"][2:end, 2:end]  # drop the b_1 NaN row/col
    labels = ["n=$n" for n in 2:13]
    p = heatmap(labels, labels, M, size=(700, 650),
        title="b_n correlation matrix ($Lstr)\n" *
              "Nearest-neighbor anticorrelation ≈ −0.5 (tridiagonal structure)",
        xlabel="", ylabel="", c=:RdBu, clims=(-1, 1),
        yflip=true, aspect_ratio=:equal)
    return p
end
savefig_at(plot_bns_corr(d20, "L=20"), "11_1_bns_corr_L20.png")
savefig_at(plot_bns_corr(d05, "L=5"), "11_2_bns_corr_L05.png")

# ────────── Plot 4.3: Bn correlation matrix ──────────
# CONCEPT: In stark contrast to bn, Bn values are all positively correlated
# > 0.87. Essentially rank-1 — all Bn's move together. This means the
# Hellmann-Bich polynomial combining the anticorrelated bn's produces an
# effectively one-factor noise model on Bn. Physical meaning: there's really
# only ~1 independent piece of information across the 12 Bn's.
function plot_Bns_corr(d, Lstr)
    M = d["Bns_corr"][2:end, 2:end]
    labels = ["n=$n" for n in 2:13]
    p = heatmap(labels, labels, M, size=(700, 650),
        title="B_n correlation matrix ($Lstr)\n" *
              "All correlations > 0.87 — Bn's move together (nearly rank-1)",
        xlabel="", ylabel="", c=:RdBu, clims=(-1, 1),
        yflip=true, aspect_ratio=:equal)
    return p
end
savefig_at(plot_Bns_corr(d20, "L=20"), "12_1_Bns_corr_L20.png")
savefig_at(plot_Bns_corr(d05, "L=5"), "12_2_Bns_corr_L05.png")

# ────────── Plot 4.4: Eigenvalue spectra of the covariance matrices ──────────
# CONCEPT: Eigendecomposition of a covariance matrix tells you the principal
# modes of variation. If variance is concentrated in one eigenvalue, the
# system is effectively "one-dimensional" in its noise. For Bn, we expect one
# dominant eigenvalue; for bn, we expect a broader spectrum because of the
# tridiagonal structure.
function plot_eigenspectra(d, Lstr)
    logQ_cov = d["logQ_cov"][2:end, 2:end]
    bns_cov = d["bns_cov"][2:end, 2:end]
    Bns_cov = d["Bns_cov"][2:end, 2:end]

    λ_logQ = sort(real.(eigvals(Symmetric(logQ_cov))), rev=true)
    λ_bns = sort(real.(eigvals(Symmetric(bns_cov))), rev=true)
    λ_Bns = sort(real.(eigvals(Symmetric(Bns_cov))), rev=true)

    # Normalize to fraction of total variance
    f_logQ = λ_logQ ./ sum(λ_logQ)
    f_bns = λ_bns ./ sum(λ_bns)
    f_Bns = λ_Bns ./ sum(λ_Bns)

    p = plot(size=(900, 600), yscale=:log10,
        xlabel="eigenvalue rank (sorted largest→smallest)",
        ylabel="fraction of total variance",
        title="Eigenvalue spectra of covariance matrices ($Lstr)\n" *
              "Bn spectrum collapses to ~1 dominant mode ⇒ effectively rank-1",
        legend=:topright)
    plot!(p, 1:length(f_logQ), f_logQ, marker=:circle, markersize=6,
        linewidth=2, label="logQ_N", color=:darkgreen)
    plot!(p, 1:length(f_bns), f_bns, marker=:square, markersize=6,
        linewidth=2, label="b_n", color=:steelblue)
    plot!(p, 1:length(f_Bns), f_Bns, marker=:diamond, markersize=6,
        linewidth=2, label="B_n", color=:crimson)
    return p
end
savefig_at(plot_eigenspectra(d20, "L=20"), "13_1_eigenspectra_L20.png")
savefig_at(plot_eigenspectra(d05, "L=5"), "13_2_eigenspectra_L05.png")

# ────────── Plot 4.5: bn nearest-neighbor anticorrelation magnitude ──────────
# CONCEPT: Pulling out the tridiagonal structure: plot the values of
# corr(b_n, b_{n+1}) across n. These should be roughly constant at ~−0.46
# to −0.5, with the b2↔b3 pair being stronger (−0.68 at L=20). That pattern
# is what the Ushcats recursion structurally produces.
function plot_nn_anticorr(d20, d05)
    ns = 2:12
    nn20 = [d20["bns_corr"][i+1, i+2] for i in ns .- 1]  # indices are 1-based
    nn05 = [d05["bns_corr"][i+1, i+2] for i in ns .- 1]

    p = plot(size=(800, 450),
        xlabel="n", ylabel="corr(b_n, b_{n+1})",
        title="Nearest-neighbor b_n anticorrelation\n" *
              "Tridiagonal noise structure — consistent across L",
        ylims=(-0.8, 0.0), legend=:bottomright)
    plot!(p, ns, nn20, marker=:circle, markersize=7, linewidth=2,
        label="L=20", color=:steelblue)
    plot!(p, ns, nn05, marker=:square, markersize=7, linewidth=2,
        label="L=5", color=:darkorange)
    hline!(p, [-0.5], linestyle=:dash, linecolor=:black, label="−1/2 reference")
    return p
end
savefig_at(plot_nn_anticorr(d20, d05), "14_bn_nn_anticorr.png")


"/Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/plots/14_bn_nn_anticorr.png"

In [7]:

# ═══════════════════════════════════════════════════════════════════════════
#  PART 5: L DEPENDENCE — signal/noise, physical content, extraction
# ═══════════════════════════════════════════════════════════════════════════

# ────────── Plot 5.1: Per-run logQ std vs N, comparing L=5 and L=20 ──────────
# CONCEPT: Per-run logQ noise is set by WL convergence criteria (flatness,
# 1/t phase, maxiter) — not by box size in any deep way. Check whether the
# two L's have similar logQ noise. Similar → bias comes from noise not physics.
function plot_logQ_noise_vs_L(d20, d05)
    Ns = 1:13
    p = plot(size=(800, 500),
        xlabel="N", ylabel="per-run std(logQ_N)",
        title="Per-run WL noise on logQ — similar at both L\n" *
              "WL convergence is V-independent → same maxiter → same noise",
        legend=:topleft)
    plot!(p, Ns, d20["logQ_std"][Ns.+1], marker=:circle, markersize=7,
        linewidth=2, label="L=20", color=:steelblue)
    plot!(p, Ns, d05["logQ_std"][Ns.+1], marker=:square, markersize=7,
        linewidth=2, label="L=5", color=:darkorange)
    return p
end
savefig_at(plot_logQ_noise_vs_L(d20, d05), "15_logQ_noise_vs_L.png")

# ────────── Plot 5.2: |b_n| magnitude comparing L=5 vs L=20 ──────────
# CONCEPT: The physical signal in bn is much larger at L=5 than at L=20
# because at large V the configuration integrals are dominated by ideal-gas
# V^N background, relegating the cluster-integral physics to exponentially
# small relative corrections. This is the "SNR vs V" tradeoff.
function plot_bn_magnitude_vs_L(d20, d05)
    ns = 2:13
    p = plot(size=(800, 600), yscale=:log10,
        xlabel="n", ylabel="|b_n| (from mean logQ)",
        title="Physical b_n magnitude is larger at smaller L\n" *
              "At large V, bn is a tiny correction to V^N ideal-gas bulk",
        legend=:topright)
    plot!(p, ns, abs.(d20["bns_from_meanlogQ"][ns]), marker=:circle,
        markersize=7, linewidth=2, label="L=20", color=:steelblue)
    plot!(p, ns, abs.(d05["bns_from_meanlogQ"][ns]), marker=:square,
        markersize=7, linewidth=2, label="L=5", color=:darkorange)
    return p
end
savefig_at(plot_bn_magnitude_vs_L(d20, d05), "16_bn_magnitude_vs_L.png")

# ────────── Plot 5.3: B_n magnitude comparing L=5 vs L=20 ──────────
# CONCEPT: B_n also depends on L — finite-periodic-box integrals differ
# slightly from infinite-V integrals. For Gaussian gas with Gaussian-decay
# Mayer function, the finite-L correction is ~exp(−(L/2)²) so L=20 ≈ infinite;
# L=5 has small (few %) finite-L corrections. This plot visualizes that.
function plot_Bn_magnitude_vs_L(d20, d05)
    ns = 2:13
    p = plot(size=(800, 600), yscale=:log10,
        xlabel="n", ylabel="B_n (from mean logQ)",
        title="B_n at two L — differs by finite-box correction to graph integrals\n" *
              "Exponentially small for Gaussian gas but nonzero at L=5",
        legend=:topleft)
    plot!(p, ns, d20["Bns_from_meanlogQ"][ns], marker=:circle, markersize=7,
        linewidth=2, label="L=20", color=:steelblue)
    plot!(p, ns, d05["Bns_from_meanlogQ"][ns], marker=:square, markersize=7,
        linewidth=2, label="L=5", color=:darkorange)
    return p
end
savefig_at(plot_Bn_magnitude_vs_L(d20, d05), "17_Bn_magnitude_vs_L.png")

# ────────── Plot 5.4: Relative SEM at both L — how well known is each Bn? ──────────
# CONCEPT: Relative SEM = SEM(Bn) / Bn. This is the fractional uncertainty
# on our best Bn estimate after 100k runs. Smaller is better. L=5 is
# competitive with L=20 despite physical signal being smaller relative to
# ideal-gas bulk (because Bn magnitude is also smaller).
function plot_relative_SEM(d20, d05)
    ns = 2:13
    rel20 = d20["Bns_sem"][ns] ./ abs.(d20["Bns_from_meanlogQ"][ns])
    rel05 = d05["Bns_sem"][ns] ./ abs.(d05["Bns_from_meanlogQ"][ns])

    p = plot(size=(800, 500), yscale=:log10,
        xlabel="n", ylabel="SEM(B_n) / |B_n|",
        title="Relative SEM of B_n at 100k runs\n" *
              "How precisely does this campaign determine each coefficient?",
        legend=:bottomright)
    plot!(p, ns, rel20, marker=:circle, markersize=7, linewidth=2,
        label="L=20", color=:steelblue)
    plot!(p, ns, rel05, marker=:square, markersize=7, linewidth=2,
        label="L=5", color=:darkorange)
    return p
end
savefig_at(plot_relative_SEM(d20, d05), "18_relative_SEM_Bn.png")

# ────────── Plot 5.5: Ratio of consecutive B_n — radius of convergence estimate ──────────
# CONCEPT: If B_n ~ C·r^(−n) asymptotically, then |B_{n+1}/B_n| → 1/r, the
# reciprocal of the radius of convergence. Also the correlation structure
# of B_n helps here: the strong positive correlation between neighbors means
# the ratio is much better-determined than the individual Bn's. This is a
# practical benefit of the correlation structure.
function plot_Bn_ratio(d20, d05)
    ns = 2:12
    r20 = [d20["Bns_from_meanlogQ"][n+1] / d20["Bns_from_meanlogQ"][n] for n in ns]
    r05 = [d05["Bns_from_meanlogQ"][n+1] / d05["Bns_from_meanlogQ"][n] for n in ns]

    p = plot(size=(800, 500),
        xlabel="n", ylabel="B_{n+1} / B_n",
        title="Consecutive-coefficient ratios (asymptotically → 1/ρ_c)\n" *
              "Strong Bn-Bn+1 correlation (>0.99) means ratios are well-determined",
        legend=:bottomright)
    plot!(p, ns, r20, marker=:circle, markersize=7, linewidth=2,
        label="L=20", color=:steelblue)
    plot!(p, ns, r05, marker=:square, markersize=7, linewidth=2,
        label="L=5", color=:darkorange)
    return p
end
savefig_at(plot_Bn_ratio(d20, d05), "19_Bn_ratio.png")


"/Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/plots/19_Bn_ratio.png"

In [8]:

# ════════════════════════════════════════════════════════════════════════════
#  OMNIBUS SUMMARY — one plot to rule them all, for the poster wall
# ════════════════════════════════════════════════════════════════════════════

# ────────── Plot 6: A single 2×3 summary grid of the key findings ──────────
summary_p = plot(size=(1500, 900), layout=(2, 3))

# (1) SEM scaling L=20
snap_ks = sort(collect(keys(d20["snapshot_sem_Bns"])))
plot!(summary_p, subplot=1, xscale=:log10, yscale=:log10,
    xlabel="k", ylabel="SEM(B_n)",
    title="SEM ∝ k^(−1/2) ✓ (L=20)", legend=false)
for n in 2:13
    sems = [d20["snapshot_sem_Bns"][k][n] for k in snap_ks]
    plot!(summary_p, subplot=1, snap_ks, sems, marker=:circle, markersize=3)
end
k_ref = snap_ks
ref_anchor = d20["snapshot_sem_Bns"][snap_ks[1]][13]
plot!(summary_p, subplot=1, k_ref, ref_anchor .* sqrt.(snap_ks[1] ./ k_ref),
    linestyle=:dash, linecolor=:black, linewidth=2)

# (2) Jensen gap / SEM vs n for both L
ns = 2:13
plot!(summary_p, subplot=2,
    xlabel="n", ylabel="|gap|/SEM",
    title="Jensen bias grows with n", legend=:topleft)
plot!(summary_p, subplot=2, ns, abs.(d20["Bns_jensen_over_sem"][ns]),
    marker=:circle, markersize=5, linewidth=2, label="L=20", color=:steelblue)
plot!(summary_p, subplot=2, ns, abs.(d05["Bns_jensen_over_sem"][ns]),
    marker=:square, markersize=5, linewidth=2, label="L=5", color=:darkorange)
hline!(summary_p, [3], linestyle=:dash, color=:black, subplot=2, label="3σ")

# (3) Per-run SNR for bn and Bn (L=20)
plot!(summary_p, subplot=3, yscale=:log10,
    xlabel="n", ylabel="|mean|/std per run",
    title="Per-run SNR (L=20)", legend=:topright)
snr_b = abs.(d20["bns_mean"][2:13]) ./ d20["bns_std"][2:13]
snr_B = abs.(d20["Bns_mean"][2:13]) ./ d20["Bns_std"][2:13]
plot!(summary_p, subplot=3, 2:13, snr_b, marker=:circle, markersize=5,
    linewidth=2, label="b_n", color=:steelblue)
plot!(summary_p, subplot=3, 2:13, snr_B, marker=:square, markersize=5,
    linewidth=2, label="B_n", color=:crimson)
hline!(summary_p, [1.0], linestyle=:dash, color=:black, subplot=3, label="")

# (4) bn corr matrix L=20
M = d20["bns_corr"][2:end, 2:end]
labels = ["$n" for n in 2:13]
heatmap!(summary_p, subplot=4, labels, labels, M,
    title="bn correlations: tridiagonal",
    c=:RdBu, clims=(-1, 1), yflip=true, aspect_ratio=:equal)

# (5) Bn corr matrix L=20
M = d20["Bns_corr"][2:end, 2:end]
heatmap!(summary_p, subplot=5, labels, labels, M,
    title="Bn correlations: rank-1",
    c=:RdBu, clims=(-1, 1), yflip=true, aspect_ratio=:equal)

# (6) Bn from mean logQ at both L
plot!(summary_p, subplot=6, yscale=:log10,
    xlabel="n", ylabel="B_n from mean logQ",
    title="Bn vs order, both L", legend=:topleft)
plot!(summary_p, subplot=6, 2:13, d20["Bns_from_meanlogQ"][2:13],
    marker=:circle, markersize=5, linewidth=2, label="L=20", color=:steelblue)
plot!(summary_p, subplot=6, 2:13, d05["Bns_from_meanlogQ"][2:13],
    marker=:square, markersize=5, linewidth=2, label="L=5", color=:darkorange)

plot!(summary_p, plot_title="Key findings from 100k benchmark campaigns",
    plot_titlefontsize=14, margin=4mm)
savefig_at(summary_p, "00_OMNIBUS_summary.png")

println("All plots saved to $OUTDIR")
println("Ordering: 00 is the summary; 01–19 walk from fundamentals through to advanced analysis.")

All plots saved to /Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/plots
Ordering: 00 is the summary; 01–19 walk from fundamentals through to advanced analysis.


In [9]:
# ────────── Plot 1.4: Mean ± SEM vs k for key observables ──────────
# CONCEPT: Direct visualization of convergence of estimates with increasing
# number of runs. Error bars shrink ~1/sqrt(k).

function plot_observable_vs_k(d, Lstr, quantity::Symbol)
    snap_ks = sort(collect(keys(d["snapshot_sem_Bns"])))

    p = plot(size=(900,600),
        xlabel="number of runs k",
        ylabel=string(quantity),
        title="Convergence of $(quantity) with error bars ($Lstr)",
        legend=:outerright)

    for n in 2:8  # keep readable
        means = Float64[]
        sems  = Float64[]

        for k in snap_ks
            if quantity == :Bn
                vals = @view d["Bns_store"][n, 1:k]
            elseif quantity == :bn
                vals = @view d["bns_store"][n, 1:k]
            elseif quantity == :logQ
                vals = @view d["logQ_store"][n, 1:k]
            elseif quantity == :Q
                logvals = @view d["logQ_store"][n, 1:k]
                vals = exp.(logvals)
            end

            push!(means, mean(vals))
            push!(sems, std(vals)/sqrt(k))
        end

        plot!(p, snap_ks, means, yerror=sems,
            marker=:circle, label="$(quantity)_$n")
    end

    return p
end

savefig_at(plot_observable_vs_k(d20, "L=20", :Bn),  "20_1_Bn_vs_k_L20.png")
savefig_at(plot_observable_vs_k(d20, "L=20", :bn),  "20_2_bn_vs_k_L20.png")
savefig_at(plot_observable_vs_k(d20, "L=20", :logQ),"20_3_logQ_vs_k_L20.png")
savefig_at(plot_observable_vs_k(d20, "L=20", :Q),   "20_4_Q_vs_k_L20.png")

savefig_at(plot_observable_vs_k(d05, "L=5", :Bn),  "20_5_Bn_vs_k_L05.png")
savefig_at(plot_observable_vs_k(d05, "L=5", :bn),  "20_6_bn_vs_k_L05.png")
savefig_at(plot_observable_vs_k(d05, "L=5", :logQ),"20_7_logQ_vs_k_L05.png")
savefig_at(plot_observable_vs_k(d05, "L=5", :Q),   "20_8_Q_vs_k_L05.png")

"/Users/mckinleypaul/Documents/montecarlo/gc_wl/data/gaussian_gas/error_investigation/plots/20_8_Q_vs_k_L05.png"

In [13]:
function format_with_uncertainty(x::Float64, err::Float64)
    if err == 0 || isnan(err)
        return @sprintf("%.6g", x)
    end

    exponent = floor(Int, log10(err))
    scale = 10.0^exponent

    err_scaled = err / scale
    err_rounded = round(err_scaled, digits=1)

    digits = max(0, -exponent + (err_rounded < 1.5 ? 1 : 0))
    x_rounded = round(x, digits=digits)

    err_digits = Int(round(err / 10.0^(-digits)))

    return @sprintf("%.*f(%d)", digits, x_rounded, err_digits)
end

function print_dual_estimator_table(d, label)
    println("\n==============================")
    println(" Virial Coefficients ($label)")
    println("==============================\n")

    # Header
    @printf("%-4s  %-20s  %-20s  %-10s\n",
        "n",
        "mean-of-runs",
        "from mean logQ",
        "Δ/SEM")

    println("---------------------------------------------------------------")

    # ───── B_n ─────
    println("\nB_n:")
    for n in 2:13
        mean_val = d["Bns_mean"][n]
        sem      = d["Bns_sem"][n]
        alt_val  = d["Bns_from_meanlogQ"][n]
        gap_sem  = d["Bns_jensen_over_sem"][n]

        formatted = format_with_uncertainty(mean_val, sem)

        @printf("B_%-2d  %-20s  %-20.10g  %+8.2f\n",
            n, formatted, alt_val, gap_sem)
    end

    # ───── b_n ─────
    println("\nb_n:")
    for n in 1:13
        mean_val = d["bns_mean"][n]
        sem      = d["bns_sem"][n]
        alt_val  = d["bns_from_meanlogQ"][n]
        gap_sem  = d["bns_jensen_over_sem"][n]

        formatted = format_with_uncertainty(mean_val, sem)

        @printf("b_%-2d  %-20s  %-20.10g  %+8.2f\n",
            n, formatted, alt_val, gap_sem)
    end

    println("\nNotes:")
    println("• mean-of-runs = E[b_n(run)] (reported with SEM)")
    println("• from mean logQ = b_n(E[logQ]) (nonlinear estimator)")
    println("• Δ/SEM = (difference) / statistical error")
    println("  → |Δ/SEM| ≫ 1 indicates real bias (Jensen effect)")
end

function print_latex_table(d, label)
    println("\n% LaTeX table for $label")
    println("\\begin{tabular}{c c c c}")
    println("\\hline")
    println("\$n\$ & mean-of-runs & from mean logQ & \$\\Delta/\\mathrm{SEM}\$ \\\\")
    println("\\hline")

    for n in 2:13
        mean_val = d["Bns_mean"][n]
        sem      = d["Bns_sem"][n]
        alt_val  = d["Bns_from_meanlogQ"][n]
        gap_sem  = d["Bns_jensen_over_sem"][n]

        formatted = format_with_uncertainty(mean_val, sem)

        @printf("%d & %s & %.6g & %.2f \\\\\n",
            n, formatted, alt_val, gap_sem)
    end

    println("\\hline")
    println("\\end{tabular}")
end

print_latex_table (generic function with 1 method)

In [15]:
print_dual_estimator_table(d20, "L=20")


 Virial Coefficients (L=20)

n     mean-of-runs          from mean logQ        Δ/SEM     
---------------------------------------------------------------

B_n:
B_2   0.49966(3)            0.4995889718             +2.72
B_3   0.99831(8)            0.9979041688             +4.90
B_4   2.4932(3)             2.491333885              +6.55
B_5   6.9742(10)            6.965734041              +8.31
B_6   20.904(4)             20.86660976              +9.98
B_7   65.644(14)            65.48364147             +11.63
B_8   213.19(5)             212.5045395             +13.27
B_9   710.2(2)              707.2879576             +14.90
B_10  2413.4(7)             2401.168402             +16.52
B_11  8334(3)               8282.465302             +18.13
B_12  29159(11)             28944.94718             +19.74
B_13  103158(42)            102266.0696             +21.34

b_n:
b_1   1                     1                        +0.00
b_2   -0.49966(3)           -0.4995889718            -2.72
b_3   0

In [16]:
print_dual_estimator_table(d05, "L=5")


 Virial Coefficients (L=5)

n     mean-of-runs          from mean logQ        Δ/SEM     
---------------------------------------------------------------

B_n:
B_2   0.49216(3)            0.4920897854             +2.69
B_3   0.93555(8)            0.9351629661             +4.91
B_4   2.2046(3)             2.202922876              +6.54
B_5   5.8012(9)             5.794071817              +8.32
B_6   16.334(3)             16.30486661             +10.01
B_7   48.152(10)            48.03179456             +11.69
B_8   146.74(4)             146.2542581             +13.36
B_9   458.55(13)            456.6284702             +15.02
B_10  1461.5(5)             1453.907046             +16.67
B_11  4733(2)               4702.895752             +18.32
B_12  15528(6)              15410.95864             +19.95
B_13  51510(21)             51051.57674             +21.59

b_n:
b_1   1                     1                        +0.00
b_2   -0.49216(3)           -0.4920897854            -2.69
b_3   0.